---
title: 'Phase 1 validation: testing a basic model'
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.5
  kernelspec:
    display_name: Python 3 (ipykernel)
    language: python
    name: python3
---


The goal of this notebook is to show that a classical-quantum model generative model can actually learn, using a simple model. For this, we would be using the torch library to create the classical part of our network, and add the quantum part through the functions and objects we created in `data.py` and `models.py`. You can find a more detailed explanation in the `docs`folder, including a schema of our model. Here, we will focus on explaining what we are doing instead of why.


In [ ]:
# First we add the src folder so the notebook can view the code inside
import sys
sys.path.append("../src")

In [ ]:
from torch.nn import ReLU, Tanh

from data import * 
from models import *

import torch
import numpy as np

# We build the model
model = nn.Sequential(
    nn.Linear(6, 6),
    nn.Tanh(),
    QuantumCircuit(),
    nn.Linear(56, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)

Now, before training the model, we have to generate the desired output so we can train using the MDD loss.


In [ ]:
desired_output = two_gaussian(256)
print(desired_output[:5])

Let's also view this in a graph to ensure that this is working.


In [ ]:
import matplotlib.pyplot as plt

points = desired_output.numpy() # tensor -> numpy array
plt.scatter(points[:, 0], points[:, 1])
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

As we can see, we have two gaussians centered around (-1, 0) and (1, 0).

Now let's test the random_input_numbers function.


In [ ]:
random_numers_test = []
for i in range(256): # Here, we make a loop instead of just calling random_input_numbers once, because of the seed. 
    # It is best to test it like a real simulation
    random_num_test = random_input_numbers(6, seed=i) # Also, we set the seed as i because taking the same seed
    # would just output 256 times the same numbers.
    random_numers_test.append(random_num_test.numpy())

all_values = np.array(random_numers_test).flatten()
plt.hist(all_values, bins=30)
plt.xlabel("value")
plt.ylabel("count")
plt.show()

As we can see here, we do have a random normal distribution, so the random_input_numbers function is properly working.

Now, before doing any training, let's see what result we get from running our model once.

In [ ]:
outputs_test = []
for i in range(256):
    x = random_input_numbers(6, seed=i)
    y = model(x)
    outputs_test.append(y)

print(outputs_test[:5])

In [ ]:
xs = [t[0, 0].item() for t in outputs_test]
ys = [t[0, 1].item() for t in outputs_test]

plt.figure(figsize=(6, 6))
plt.scatter(xs, ys, alpha=0.6, s=15)
plt.xlabel("Output dim 0")
plt.ylabel("Output dim 1")
plt.title("Model outputs for 256 random inputs")
plt.grid(True, alpha=0.3)
plt.show()

As we can see, this is what our untrained model outputs. It looks nothing like the two gaussians we want. Now let's train the model.






We can actually optimize the compute time by sending straight 256 times 6 random numbers into the 
